#This example showcases getting a word count for a book.

In [ ]:
#this next note book: we will find sort customer ID and total spent in decending order from customer-orders.csv
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('MinTemps')
sc = SparkContext(conf = conf)

def parseLine(line):
    fields = line.split(',')
    id = fields[0]
    orderPrice = float(fields[2])
    return (id, orderPrice)

lines = sc.textFile('./customer-orders.csv')
parsedLines = lines.map(parseLine)
customerTotal = parsedLines.reduceByKey(lambda x, y: x + y).sortBy(lambda x: x[1], ascending=False)

table = PrettyTable()
table.field_names = ["ID", "Total"]

for (id, total) in customerTotal.collect():
    table.add_row([id, f"{total:.2f}"])

print(table)

In [ ]:
## this next note book: Data Reading from fakefriends.csv
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable

conf = SparkConf().setAppName("Friends").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

friends_file = sc.textFile("./fakefriends.csv")

table = PrettyTable()
table.field_names = ["Friend ID", "Friend Name", "Age", "Number of Friends"]

for line in friends_file.collect():
    fields = line.split(",")
    table.add_row([fields[0], fields[1], fields[2], fields[3]])


print(table)


In [2]:
## this next note book: will data 
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable
import re


conf = SparkConf().setAppName("WordCount").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

WORD_RE = re.compile(r"\b\w+(?:`\w+)*\b", re.UNICODE)

def normalize_word(line):
    return WORD_RE.findall(line.lower())

book = sc.textFile("./pride_and_prejudice.txt")
words = book.flatMap(normalize_word)
word_counts = words.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)
word_counts_sorted = word_counts.sortBy(lambda x: x[1], ascending=False).collect()


table = PrettyTable()
table.field_names = ["Word", "Count"]

for word, count in word_counts_sorted:
    table.add_row([word, count])

print(table)


+-------------------+-------+
|        Word       | Count |
+-------------------+-------+
|        the        |  4846 |
|         to        |  4405 |
|         of        |  3962 |
|        and        |  3835 |
|        her        |  2260 |
|         i         |  2098 |
|         a         |  2094 |
|         in        |  2051 |
|        was        |  1874 |
|        she        |  1732 |
|        that       |  1620 |
|         it        |  1603 |
|        not        |  1520 |
|        you        |  1417 |
|         he        |  1350 |
|        his        |  1289 |
|         be        |  1280 |
|         as        |  1239 |
|        had        |  1181 |
|        with       |  1149 |
|        for        |  1118 |
|        but        |  1040 |
|         is        |  945  |
|        have       |  883  |
|         at        |  831  |
|         mr        |  807  |
|        him        |  765  |
|         on        |  745  |
|         by        |  724  |
|         my        |  710  |
|         

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("Learning").master("local[*]").getOrCreate()
sc = SparkContext.getOrCreate(conf=conf)

data_py = spark.read.csv("./fakefriends.csv", header=True, inferSchema=True).toDF("id", "name", "age", "num_friends")

print(data_py.show())




In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("Simple Spark Presentation Demo").master("local[*]").getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Spark is ready")

In [ ]:
df = spark.createDataFrame([("Alice", 30), ("Bob", 25), ("Charlie", 35)],["name", "age"])

# Transformation - it creates a new DataFrame called adults
adults = df.filter(F.col("age") >= 30)

# Action
adults.show()

In [ ]:
numbers = spark.sparkContext.parallelize([1, 2, 3, 4, 5, 6], 2)

# Narrow transformation
#I create a small RDD with numbers from 1 to 6 and split it into two partitions
even_numbers = numbers.filter(lambda x: x % 2 == 0)

even_numbers.collect()

In [ ]:
pairs = spark.sparkContext.parallelize([("A", 1), ("B", 2), ("A", 3),("B", 4)], 2)

# Wide transformation
totals = pairs.reduceByKey(lambda x, y: x + y)

totals.collect()

In [ ]:
#This cell creates a small DataFrame and saves it as a Parquet file.
people = spark.createDataFrame([(1, "Alice", 30, "Sales"),(2, "Bob", 25, "IT"),(3, "Charlie", 35, "IT")], ["id", "name", "age", "department"])

#Then I read the Parquet file back into Spark
people.write.mode("overwrite").parquet("people_parquet")

#Parquet
parquet_df = spark.read.parquet("people_parquet")

parquet_df.show()
parquet_df.printSchema()

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("friends", IntegerType(), True)
])

friends_df = spark.read.schema(schema).csv("fakefriends.csv")

name_age_df = friends_df.select("name", "age")

name_age_df.show(20, truncate=False)

In [ ]:
from pyspark.sql import SparkSession, functions as func


spark = SparkSession.builder.appName("WordCount").getOrCreate()

inputDF = spark.read.text("./pride_and_prejudice.txt")
words = inputDF.select(func.explode(func.split(inputDF.value, r'\W+')).alias('word'))
wordsWithoutEmptyString = words.filter(words.word != "")
lowerCaseWords = wordsWithoutEmptyString.select(func.lower(wordsWithoutEmptyString.word).alias('word'))
result = lowerCaseWords.groupBy('word').count().sort('count', ascending=False)

result.show()

spark.stop()



In [ ]:
from pyspark.sql import SparkSession, functions as fun
from pyspark.sql.types import StructType, StructField, StringType, IntegerType ,FloatType

spark = SparkSession.builder.appName("MinTemp").getOrCreate()

schema = StructType({
    StructField("StationID", StringType()),
    StructField("Date", IntegerType()),
    StructField("Type", StringType()),
    StructField("Value", FloatType()),
    
})

df = spark.read.csv(
    './weatherData1800s.csv',
    schema=schema,
    header=False
)

df.printSchema()

minTemp = df.filter(df.type == 'TMIN')
stationTemp = minTemp.select()

In [ ]:
from pyspark.sql import SparkSession, functions as fun
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

# Create SparkSession
spark = SparkSession.builder.appName("CustomerSpend").getOrCreate()

# Define the schema for customer-orders.csv
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("order_id", IntegerType()),
    StructField("amount", FloatType())
])

# Read CSV file into a DataFrame
df = spark.read.csv(
    "customer-orders.csv",
    schema=schema,
    header=False
)

# Check schema
#df.printSchema()

# Group by customer_id, add up amount, then sort in desc
customer_totals = df.groupBy("customer_id").agg(fun.round(fun.sum("amount"), 1).alias("total_spent")).orderBy(fun.desc("total_spent"))


# Show result
customer_totals.show()

In [ ]:
from pyspark.sql import SparkSession, functions as fun
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

# Create SparkSession
spark = SparkSession.builder.appName("CustomerSpend").getOrCreate()

# Define the schema for customer-orders.csv
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("order_id", IntegerType()),
    StructField("amount", FloatType())
])

# Read CSV file into a DataFrame
df = spark.read.csv(
    "customer-orders.csv",
    schema=schema,
    header=False
)

# Check schema
#df.printSchema()

# Group by customer_id, add up amount, then sort in desc
customer_totals = df.groupBy("customer_id").agg(fun.round(fun.sum("amount"), 1).alias("total_spent")).orderBy(fun.desc("total_spent"))


# Show result
customer_totals.show()

In [ ]:
# this is the same as above but we use spark sql
from pyspark.sql import SparkSession, functions as fun
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

# Create SparkSession
spark = SparkSession.builder.appName("CustomerSpendSQL").getOrCreate()

# Define the schema for customer-orders.csv
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("order_id", IntegerType()),
    StructField("amount", FloatType())
])

# Read CSV file into a DataFrame
df = spark.read.csv(
    "customer-orders.csv",
    schema=schema,
    header=False
)

#
df.createGlobalTempView('orders')

customers = spark.sql ("""
    SELECT 
        customer_id, 
        ROUND(SUM(amount), 1) AS total_spent
    FROM global_temp.orders
    GROUP BY customer_id 
    ORDER BY total_spent DESC

"""
)

# Show result
customers.show()

In [ ]:
import os

# Set before importing pyspark.pandas
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = (
    SparkSession.builder
    .appName("Pandas_On_Spark")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

ps_df = ps.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", "Charlie", "David", "Emma"],
    "age": [24, 55, 33, 54, 29],
    "salary": [5000, 4000, 4444, 3333, 2222]
})

print("Original pandas-on-Spark DataFrame:")
print(ps_df)

# Add a new column: salary after 10% promotion
ps_df["salary_after_promotion"] = ps_df["salary"] * 1.1

print("\nAfter promotions applied:")
print(ps_df)

# Filter people over age 30
filtered_ps_df = ps_df[ps_df["age"] > 30]

print("\nDataFrame with people over 30:")
print(filtered_ps_df)

# Convert pandas-on-Spark DataFrame to regular Spark DataFrame
spark_df = ps_df.to_spark()


print("\nConverted to regular Spark DataFrame:")
spark_df.show()

# Convert Spark DataFrame back to pandas-on-Spark DataFrame
ps_df_from_spark = ps.DataFrame(spark_df)

print("\nReconverted back to pandas-on-Spark DataFrame:")
print(ps_df_from_spark)

In [ ]:
import os

# Set before importing pyspark.pandas
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
import pandas as pd
import pyspark.pandas as ps

spark = (
    SparkSession.builder
    .appName("Pandas_On_Spark")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

pandas_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", "Charlie", "David", "Emma"],
    "age": [24, 55, 33, 54, 29],
    "salary": [5000, 4000, 4444, 3333, 2222]
})

print(pandas_df)

#taking a panda data frame and creating a spark DF from it
spark_df =  spark.createDataFrame(pandas_df)

#convert spark df back to pandas data frame
pd_df = spark_df.toPandas()

#create pandas-on-spark df to a spark df
ps_df = ps.DataFrame(pd_df)

ps_df['age_in_10_yesrs'] = ps_df['age'].transform(lambda x: x + 10)

converted_sparkdf = ps_df.to_spark(index_col='index')
converted_sparkdf.show()






In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, LongType
from pyspark.sql.functions import count, desc


spark = SparkSession.builder.appName("MostPopularMovies").getOrCreate()
spark.sparkContext.setLogLevel("WARN")


schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", LongType(), True)
])


moviesDF = spark.read.option("sep", "\t").schema(schema).csv("./ml-100k/u.data")

topMovieIDs = (moviesDF.groupBy("movie_id").agg(count("*").alias("rating_count")).orderBy(desc("rating_count")))

topMovieIDs.show(10)

spark.stop()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, count, desc


spark = SparkSession.builder.appName("LogInformation").getOrCreate()
spark.sparkContext.setLogLevel("WARN")


# Read the access log file
logsDF = spark.read.text("./access_log.txt")


# Apache log pattern
# Group 1 = IP address
# Group 2 = HTTP method
# Group 3 = endpoint
# Group 4 = HTTP protocol
# Group 5 = status code
log_pattern = r'^(\S+) \S+ \S+ \[[^\]]+\] "(\S+) ([^ ]+) ([^"]+)" (\d{3})'


# Extract useful columns from the raw log line
parsedLogsDF = logsDF.select(
    regexp_extract(col("value"), log_pattern, 1).alias("ip_address"),
    regexp_extract(col("value"), log_pattern, 2).alias("method"),
    regexp_extract(col("value"), log_pattern, 3).alias("endpoint"),
    regexp_extract(col("value"), log_pattern, 4).alias("protocol"),
    regexp_extract(col("value"), log_pattern, 5).cast("int").alias("status")
)

# Remove rows that did not match the regex
cleanLogsDF = parsedLogsDF.filter(col("ip_address") != "")


# 1. List IP addresses + sum/count of their requests
ipCounts = (
    cleanLogsDF
    .groupBy("ip_address")
    .agg(count("*").alias("request_count"))
    .orderBy(desc("request_count"))
)


# 2. List requested endpoints + count of how many times they were called
endpointCounts = (
    cleanLogsDF
    .groupBy("endpoint")
    .agg(count("*").alias("endpoint_count"))
    .orderBy(desc("endpoint_count"))
)


# 3. List HTTP status codes + count from all requests
statusCounts = (
    cleanLogsDF
    .groupBy("status")
    .agg(count("*").alias("status_count"))
    .orderBy(desc("status_count"))
)


print("Top IP addresses by request count")
ipCounts.show(10, truncate=False)

print("Top requested endpoints")
endpointCounts.show(10, truncate=False)

print("HTTP status code counts")
statusCounts.show(truncate=False)